# EX_12 — Contexto largo y multimodal (ejercicios)

**Notebook de referencia:** `notebook/12_Modelos_Contexto_Multimodales.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Presupuesto de tokens

Estima (orden de magnitud) tokens para: 10 páginas de texto, 1 imagen 1024×1024 en un modelo que la trata como patches, y 2 minutos de audio crudo 16 kHz 16-bit. Usa markdown con supuestos explícitos.


_Supuestos y estimaciones:_

Para calcular el orden de magnitud del consumo de tokens, nos basaremos en las arquitecturas estándar de la industria (como OpenAI, Anthropic o Google Gemini):Texto (10 páginas):Supuesto: Una página estándar de texto contiene aproximadamente 500 palabras.Supuesto: El factor de conversión promedio para el español/inglés en tokenizadores modernos es de aproximadamente 1 palabra $\approx$ 1.3 tokens (o 1 token $\approx$ 4 caracteres).Cálculo: $10 \text{ páginas} \times 500 \text{ palabras/página} \times 1.3 \text{ tokens/palabra} = 6,500 \text{ tokens}$.Orden de magnitud: $\sim 10^3 - 10^4$ tokens.Imagen (1024×1024 en parches/patches):Supuesto: Modelos como GPT-4V dividen las imágenes de alta resolución en parches de $512 \times 512$ píxeles. Una imagen de $1024 \times 1024$ se divide en $2 \times 2 = 4$ parches.Supuesto: Cada parche cuesta 170 tokens, más un costo base fijo de 85 tokens por la imagen completa ("low res master token").Cálculo: $85 + (4 \times 170) = 765 \text{ tokens}$. (Nota: En otros modelos como Gemini o Claude, el costo fijo por imagen puede oscilar de forma fija entre 258 y 1600 tokens independientemente de los patches).Orden de magnitud: $\sim 10^3$ tokens.Audio (2 minutos de audio crudo 16 kHz 16-bit):Supuesto: Modelos nativos de audio (como Gemini 1.5) suelen tokenizar el audio basándose en el tiempo de duración, consumiendo aproximadamente entre 20 y 50 tokens por segundo de audio (dependiendo de la tasa de compresión del encoder latente). Tomemos una media de 30 tokens/segundo.Cálculo: $2 \text{ minutos} \times 60 \text{ segundos/minuto} \times 30 \text{ tokens/segundo} = 3,600 \text{ tokens}$.Orden de magnitud: $\sim 10^3$ tokens.Resumen del presupuesto total: El total combinado de los tres elementos rondará los 10,000 - 11,000 tokens, lo cual es un tamaño sumamente ligero para cualquier ventana de contexto moderna.


## Actividad 2 — Estrategia de ventana

Describe cómo partirías un documento de 200k tokens para un modelo de 128k de ventana (resumen jerárquico, índice, etc.). Respuesta en español.


_Estrategia:_

Estrategia:Dado que el documento tiene 200k tokens y la ventana de contexto máxima del modelo es de 128k tokens, el archivo supera el límite físico y no puede procesarse de un solo golpe directamente de forma nativa. Para solucionarlo, implementaría una Estrategia de Resumen Jerárquico combinada con un Índice de Recuperación (RAG):Fragmentación (Chunking) con solapamiento: Dividir el documento original de 200k tokens en bloques más pequeños (por ejemplo, fragmentos de 4,000 tokens cada uno, con un solapamiento de 500 tokens para no perder el contexto de transición entre fragmentos). Esto nos daría unos 50-60 fragmentos.Resumen en paralelo (Map): Enviar cada fragmento al modelo de manera independiente para generar un resumen ejecutivo condensado de cada bloque (reduciendo cada bloque a unos 400 tokens).Generación del Contexto Maestro / Índice (Reduce): Concatenar todos los mini-resúmenes en un único documento estructurado. 60 fragmentos $\times$ 400 tokens resultan en un "Índice/Mapa Global" de unos 24k tokens.Inyección en la ventana final: * Al realizar una consulta, inyectamos en la ventana de 128k el Índice Global (24k tokens) para que el modelo entienda la estructura completa del documento.Usamos técnicas de embeddings (RAG) o búsquedas semánticas previas para extraer únicamente los 3 o 4 fragmentos originales completos de 4k tokens que sean más relevantes para la pregunta del usuario (aprox. 16k tokens).De este modo, consumimos un total estimado de $24k + 16k = 40k \text{ tokens}$, dejando más de la mitad de la ventana de 128k libre para la respuesta del usuario y el historial de la conversación.


## Actividad 3 — API multimodal (stub)

Si tu curso usa un proveedor con visión, deja un **stub** que construya `messages` con una imagen (URL o path) + pregunta. Si no, comenta el formato esperado (`image_url`, etc.).


In [ ]:
# TODO: multimodal message structure (pseudo)
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "¿Qué puedes decirme sobre los datos financieros que se muestran en este gráfico?"
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://example.com/path/to/image_336f19.png",
                    "detail": "high"  # Opciones: "low", "high" o "auto"
                }
            }
        ]
    }
]